<a href="https://colab.research.google.com/github/miguelmccormickudg/Gravedad-Territorial-Relativista/blob/main/red_neural_alternativa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Bloque 1 adaptado: red para masa m_j (regresión unitaria, MLP)

# pip install torch torchvision torchaudio scikit-learn pandas numpy

# -*- coding: utf-8 -*-
import os, random
import numpy as np
import pandas as pd
from typing import List
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# --------- 0) Reproducibilidad ----------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

# --------- 1) Carga y selección de variables desde tu archivo real ----------
# Archivo principal de Ajijic con masa y fuerza
df = pd.read_excel("resultados_gravedad_propiedades_v2.xlsx")

# Crear columnas lat/lon cómodas para uso posterior
df["lat"] = df["coordinate/latitude"]
df["lon"] = df["coordinate/longitude"]

# Convertir canInstantBook a binario 0/1
df["canInstantBook"] = (
    df["canInstantBook"]
    .map({True: 1, False: 0, "True": 1, "False": 0})
    .fillna(0)
    .astype(int)
)

# Definimos un conjunto de features numéricos que sí existen en tu archivo:
feature_cols = [
    "Reviews",
    "host_rating",
    "host_rating_Count",
    "timeAsHost_months",
    "timeAsHost_years",
    "maxGuestCapacity",
    "petsAllowed",
    "Cleanliness",
    "Accuracy",
    "Check-in",
    "Communication",
    "Location",
    "reviewsCount",
    "starRating",
    "Superhost",
    "canInstantBook",
]

# Target: masa gravitacional en log
df["log_masa"] = np.log1p(df["Masa"].clip(lower=1e-6))
target_col = "log_masa"

# Limpieza mínima
df_model = df.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)

X = df_model[feature_cols].values.astype(np.float32)
y = df_model[target_col].values.astype(np.float32).reshape(-1, 1)

# Escalado (guardamos scaler para inferencia)
scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

# --------- 2) Dataset y modelo ----------
class ListingsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: List[int] = [128, 64], pdrop=0.1):
        super().__init__()
        layers = []
        dims = [in_dim] + hidden
        for a, b in zip(dims[:-1], dims[1:]):
            layers += [nn.Linear(a, b), nn.ReLU(), nn.Dropout(pdrop)]
        layers += [nn.Linear(dims[-1], 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

# --------- 3) Entrenamiento con Early Stopping y K-Fold ----------
def train_one_fold(Xtr, ytr, Xva, yva,
                   lr=1e-3, wd=1e-4,
                   epochs=200, patience=20,
                   batch=128):
    train_ds = ListingsDataset(Xtr, ytr)
    val_ds   = ListingsDataset(Xva, yva)
    train_loader = DataLoader(train_ds, batch_size=batch, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch, shuffle=False)

    model = MLP(in_dim=Xtr.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.L1Loss()  # MAE

    best_val = float("inf"); best_state = None; wait = 0
    for ep in range(1, epochs+1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()

        # validación
        model.eval()
        val_preds = []
        val_targs = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pv = model(xb)
                val_preds.append(pv.cpu().numpy())
                val_targs.append(yb.cpu().numpy())
        val_preds = np.vstack(val_preds)
        val_targs = np.vstack(val_targs)
        val_mae = mean_absolute_error(val_targs, val_preds)

        if val_mae < best_val:
            best_val = val_mae
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_val

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_mae, fold_r2 = [], []
models = []

for fold, (tr, va) in enumerate(kf.split(X), 1):
    Xtr, ytr = X[tr], y[tr]
    Xva, yva = X[va], y[va]
    model, best_val = train_one_fold(Xtr, ytr, Xva, yva)
    models.append(model)

    # Evaluación en el fold
    with torch.no_grad():
        pred = models[-1](torch.tensor(Xva, dtype=torch.float32).to(device)).cpu().numpy()
    mae = mean_absolute_error(yva, pred)
    r2  = r2_score(yva, pred)
    fold_mae.append(mae); fold_r2.append(r2)
    print(f"Fold {fold}: MAE={mae:.4f}, R2={r2:.4f}")

print(f"CV-MAE={np.mean(fold_mae):.4f} ± {np.std(fold_mae):.4f}")
print(f"CV-R2 ={np.mean(fold_r2):.4f} ± {np.std(fold_r2):.4f}")

# --------- 4) Entrena modelo final en todo el set y guarda artefactos ----------
final_model = MLP(in_dim=X.shape[1]).to(device)
opt = torch.optim.AdamW(final_model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.L1Loss()

ds = ListingsDataset(X, y)
dl = DataLoader(ds, batch_size=128, shuffle=True)

best_loss = float("inf"); best_state = None; wait=0
for ep in range(1, 301):
    final_model.train()
    ep_losses = []
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        pred = final_model(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        ep_losses.append(loss.item())
    cur = np.mean(ep_losses[-10:]) if len(ep_losses)>=10 else np.mean(ep_losses)
    if cur < best_loss:
        best_loss = cur; best_state = final_model.state_dict(); wait = 0
    else:
        wait += 1
        if wait >= 30:
            break

final_model.load_state_dict(best_state)

os.makedirs("artefactos", exist_ok=True)
torch.save({
    "model_state": final_model.state_dict(),
    "scaler_mean": scaler.mean_,
    "scaler_scale": scaler.scale_,
    "feature_cols": feature_cols,
    "target_col": target_col
}, "artefactos/modelo_masa.pt")

print("Modelo y scaler guardados en artefactos/modelo_masa.pt")

# --------- 5) Exportar embeddings intermedios para recomendación ----------
def extract_embeddings(model: nn.Module, X: np.ndarray) -> np.ndarray:
    # toma la penúltima capa como embedding
    layers = list(model.net.children())
    # quitar última capa Linear(64->1)
    body = nn.Sequential(*layers[:-1]).to(device)
    with torch.no_grad():
        Z = body(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy()
    return Z

Z = extract_embeddings(final_model, X)
np.save("artefactos/embeddings_propiedades.npy", Z)

# Exportar ids alineados con embeddings
df_model[["id"]].to_csv("artefactos/ids_propiedades.csv", index=False)
print("Embeddings guardados para búsquedas por similitud.")


Usando dispositivo: cpu
Fold 1: MAE=0.0714, R2=0.9862
Fold 2: MAE=0.0869, R2=0.9363
Fold 3: MAE=0.0904, R2=0.9811
Fold 4: MAE=0.0824, R2=0.9816
Fold 5: MAE=0.0884, R2=0.9652
CV-MAE=0.0839 ± 0.0068
CV-R2 =0.9701 ± 0.0183
Modelo y scaler guardados en artefactos/modelo_masa.pt
Embeddings guardados para búsquedas por similitud.


In [ ]:
# Bloque 2 adaptado: red para atracción A_ij (pares (i,j))

# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

# Asume que:
# 1) Ya ejecutaste el Bloque 1 en el mismo entorno
#    y tienes df, feature_cols, final_model, scaler.
# 2) Existen:
#    - artefactos/embeddings_propiedades.npy
#    - artefactos/ids_propiedades.csv
#
# Además, necesitas un archivo pairs.csv con columnas al menos:
# id_i, id_j, c_ij_minutos, A_ij
# y opcionalmente similaridad_semantica.

# Cargar df principal (por si este bloque se ejecuta por separado)
df = pd.read_excel("resultados_gravedad_propiedades_v2.xlsx")
df["lat"] = df["coordinate/latitude"]
df["lon"] = df["coordinate/longitude"]

# Debe ser coherente con el Bloque 1
feature_cols = [
    "Reviews",
    "host_rating",
    "host_rating_Count",
    "timeAsHost_months",
    "timeAsHost_years",
    "maxGuestCapacity",
    "petsAllowed",
    "Cleanliness",
    "Accuracy",
    "Check-in",
    "Communication",
    "Location",
    "reviewsCount",
    "starRating",
    "Superhost",
    "canInstantBook",
]

# Cargar scaler desde el modelo de masa
checkpoint = torch.load("artefactos/modelo_masa.pt", map_location=device, weights_only=False)
scaler_mean = checkpoint["scaler_mean"]
scaler_scale = checkpoint["scaler_scale"]

# Cargar embeddings e ids
Z = np.load("artefactos/embeddings_propiedades.npy")
ids_df = pd.read_csv("artefactos/ids_propiedades.csv")
ids = ids_df["id"].astype(str).values
id2idx = {str(i): k for k, i in enumerate(ids)}

# 1) Cargar pares
pairs = pd.read_csv("pairs.csv").dropna(subset=["id_i", "id_j", "c_ij_minutos", "A_ij"])

# Unir atributos de j desde df_model (para evitar NaNs)
# El df_model es la versión limpia del df original y es accesible desde el Bloque 1
attrs = df_model.set_index("id")[feature_cols]
pairs = pairs.join(attrs, on="id_j", how="inner", rsuffix="_j")

# 2) Construir X_pair: [emb_i, emb_j, c_ij, similitud, atributos_j_escalados]

# Filtrar pares que estén en los embeddings
mask_valid = (
    pairs["id_i"].astype(str).isin(id2idx)
    & pairs["id_j"].astype(str).isin(id2idx)
)
pairs = pairs[mask_valid].reset_index(drop=True)

Zi = Z[[id2idx[str(i)] for i in pairs["id_i"].astype(str)]]
Zj = Z[[id2idx[str(j)] for j in pairs["id_j"].astype(str)]]

# fricción y similitud
cij = pairs["c_ij_minutos"].values.astype(np.float32).reshape(-1, 1)
if "similaridad_semantica" in pairs.columns:
    sim = pairs["similaridad_semantica"].values.astype(np.float32).reshape(-1, 1)
else:
    sim = np.zeros_like(cij, dtype=np.float32)

# atributos de j escalados con el mismo scaler de features
Xj_raw = pairs[feature_cols].values.astype(np.float32)
Xj = ((Xj_raw - scaler_mean) / scaler_scale).astype(np.float32)

X_pair = np.concatenate([Zi, Zj, cij, sim, Xj], axis=1)
y_pair = pairs["A_ij"].values.astype(np.float32).reshape(-1, 1)

class PairDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class PairNet(nn.Module):
    def __init__(self, in_dim, hidden=[256, 128], pdrop=0.1):
        super().__init__()
        layers = []
        dims = [in_dim] + hidden
        for a, b in zip(dims[:-1], dims[1:]):
            layers += [nn.Linear(a, b), nn.ReLU(), nn.Dropout(pdrop)]
        layers += [nn.Linear(dims[-1], 1)]
        self.f = nn.Sequential(*layers)
    def forward(self, x):
        return self.f(x)

ds = PairDataset(X_pair, y_pair)
dl = DataLoader(ds, batch_size=256, shuffle=True)

pairnet = PairNet(in_dim=X_pair.shape[1]).to(device)
opt = torch.optim.AdamW(pairnet.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.SmoothL1Loss()  # Huber

best = float("inf"); best_state = None; wait = 0
for ep in range(1, 251):
    pairnet.train()
    ep_loss = []
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        pred = pairnet(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        ep_loss.append(loss.item())
    cur = np.mean(ep_loss)
    print(f"Epoch {ep}: loss={cur:.4f}")
    if cur < best:
        best = cur
        best_state = pairnet.state_dict()
        wait = 0
    else:
        wait += 1
        if wait >= 25:
            break

pairnet.load_state_dict(best_state)
os.makedirs("artefactos", exist_ok=True)
torch.save(
    {
        "state": pairnet.state_dict(),
        "in_dim": X_pair.shape[1],
        "feature_cols": feature_cols
    }, "artefactos/modelo_atraccion.pt")
print("Modelo de atracción guardado en artefactos/modelo_atraccion.pt")

Epoch 1: loss=79.0571
Epoch 2: loss=42.4870
Epoch 3: loss=39.0566
Epoch 4: loss=37.3644
Epoch 5: loss=36.2961
Epoch 6: loss=35.4961
Epoch 7: loss=34.8173
Epoch 8: loss=34.2820
Epoch 9: loss=33.7347
Epoch 10: loss=33.2076
Epoch 11: loss=32.7272
Epoch 12: loss=32.1876
Epoch 13: loss=31.7088
Epoch 14: loss=31.3056
Epoch 15: loss=30.9551
Epoch 16: loss=30.7537
Epoch 17: loss=30.4989
Epoch 18: loss=30.2240
Epoch 19: loss=30.1105
Epoch 20: loss=29.9293
Epoch 21: loss=29.7475
Epoch 22: loss=29.6513
Epoch 23: loss=29.5295
Epoch 24: loss=29.3504
Epoch 25: loss=29.2220
Epoch 26: loss=29.1004
Epoch 27: loss=29.0438
Epoch 28: loss=28.9317
Epoch 29: loss=28.8406
Epoch 30: loss=28.7247
Epoch 31: loss=28.6483
Epoch 32: loss=28.5622
Epoch 33: loss=28.4324
Epoch 34: loss=28.2345
Epoch 35: loss=28.2164
Epoch 36: loss=28.0790
Epoch 37: loss=27.9001
Epoch 38: loss=27.7157
Epoch 39: loss=27.6062
Epoch 40: loss=27.4673
Epoch 41: loss=27.3796
Epoch 42: loss=27.2479
Epoch 43: loss=27.2188
Epoch 44: loss=27.11

In [ ]:
# Bloque 3 adaptado: Script territorial unificado con salida KML por clases

# -*- coding: utf-8 -*-
import os
import random
from typing import List

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ===========================================================
# 0. Configuración básica
# ===========================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

# ===========================================================
# 1. Carga de datos y selección de variables
# ===========================================================

# Usamos el archivo real con masa y atributos
df = pd.read_excel("resultados_gravedad_propiedades_v2.xlsx")

# Crear columnas lat/lon
df["lat"] = df["coordinate/latitude"]
df["lon"] = df["coordinate/longitude"]

# Convertir canInstantBook a 0/1
df["canInstantBook"] = (
    df["canInstantBook"]
    .map({True: 1, False: 0, "True": 1, "False": 0})
    .fillna(0)
    .astype(int)
)

# Definimos features coherentes con el Bloque 1
feature_cols = [
    "Reviews",
    "host_rating",
    "host_rating_Count",
    "timeAsHost_months",
    "timeAsHost_years",
    "maxGuestCapacity",
    "petsAllowed",
    "Cleanliness",
    "Accuracy",
    "Check-in",
    "Communication",
    "Location",
    "reviewsCount",
    "starRating",
    "Superhost",
    "canInstantBook",
]

# Target: log de la masa calculada
df["log_masa"] = np.log1p(df["Masa"].clip(lower=1e-6))
target_col = "log_masa"

# Para la masa actual usamos directamente la masa del modelo de gravedad
df["masa_actual"] = df["Masa"].astype(np.float32)

df_model = df.dropna(subset=feature_cols + [target_col, "lat", "lon"]).reset_index(drop=True)

X_raw = df_model[feature_cols].values.astype(np.float32)
y = df_model[target_col].values.astype(np.float32).reshape(-1, 1)

scaler = StandardScaler()
X = scaler.fit_transform(X_raw).astype(np.float32)

# ===========================================================
# 2. Dataset y red neuronal para masa m_j
# ===========================================================

class ListingsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: List[int] = [128, 64], pdrop: float = 0.1):
        super().__init__()
        layers = []
        dims = [in_dim] + hidden
        for a, b in zip(dims[:-1], dims[1:]):
            layers.append(nn.Linear(a, b))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(pdrop))
        layers.append(nn.Linear(dims[-1], 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

# ===========================================================
# 3. Entrenamiento con validación cruzada y early stopping
# ===========================================================

def train_one_fold(Xtr, ytr, Xva, yva,
                   lr=1e-3, wd=1e-4,
                   epochs=200, patience=20,
                   batch_size=128):
    train_ds = ListingsDataset(Xtr, ytr)
    val_ds = ListingsDataset(Xva, yva)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = MLP(in_dim=Xtr.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.L1Loss()  # MAE

    best_val = float("inf")
    best_state = None
    wait = 0

    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()

        model.eval()
        val_preds = []
        val_targs = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pv = model(xb)
                val_preds.append(pv.cpu().numpy())
                val_targs.append(yb.cpu().numpy())
        val_preds = np.vstack(val_preds)
        val_targs = np.vstack(val_targs)
        val_mae = mean_absolute_error(val_targs, val_preds)

        if val_mae < best_val:
            best_val = val_mae
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_val

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_mae, fold_r2 = [], []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
    Xtr, ytr = X[tr_idx], y[tr_idx]
    Xva, yva = X[va_idx], y[va_idx]
    model, best_val = train_one_fold(Xtr, ytr, Xva, yva)

    with torch.no_grad():
        preds = model(torch.tensor(Xva, dtype=torch.float32).to(device)).cpu().numpy()
    mae = mean_absolute_error(yva, preds)
    r2 = r2_score(yva, preds)
    fold_mae.append(mae)
    fold_r2.append(r2)
    print(f"Fold {fold}: MAE={mae:.4f}, R2={r2:.4f}")

print(f"CV MAE medio={np.mean(fold_mae):.4f} ± {np.std(fold_mae):.4f}")
print(f"CV R2 medio ={np.mean(fold_r2):.4f} ± {np.std(fold_r2):.4f}")

# ===========================================================
# 4. Entrenamiento final en todo el dataset
# ===========================================================

final_model = MLP(in_dim=X.shape[1]).to(device)
opt = torch.optim.AdamW(final_model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.L1Loss()
ds_all = ListingsDataset(X, y)
dl_all = DataLoader(ds_all, batch_size=128, shuffle=True)

best_loss = float("inf")
best_state = None
wait = 0

for ep in range(1, 301):
    final_model.train()
    ep_losses = []
    for xb, yb in dl_all:
        xb, yb = xb.to(device), yb.to(device)
        pred = final_model(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        ep_losses.append(loss.item())
    cur_loss = np.mean(ep_losses)
    print(f"Epoch {ep}: loss={cur_loss:.4f}")
    if cur_loss < best_loss:
        best_loss = cur_loss
        best_state = final_model.state_dict()
        wait = 0
    else:
        wait += 1
        if wait >= 30:
            break

final_model.load_state_dict(best_state)
print("Modelo final de masa entrenado")

os.makedirs("artefactos", exist_ok=True)
torch.save(
    {
        "model_state": final_model.state_dict(),
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
        "feature_cols": feature_cols,
        "target_col": target_col,
    },
    "artefactos/modelo_masa.pt",
)
print("Modelo de masa guardado en artefactos/modelo_masa.pt")

# ===========================================================
# 5. Funciones para campo de gravedad y predicción futura
# ===========================================================

def compute_gravity_field(grid_xy, prop_xy, masses, beta=0.12):
    """
    Calcula G(x) en cada punto del grid como
    G(x) = sum_j m_j * exp(-beta * d_ij)
    usando distancia Haversine en km.
    grid_xy : array (Ncells, 2) lat, lon
    prop_xy : array (Nprops, 2) lat, lon
    masses  : array (Nprops,)
    beta    : parámetro de decaimiento espacial
    """

    grid_rad = np.radians(grid_xy)
    prop_rad = np.radians(prop_xy)

    def haversine(point, others):
        dlat = others[:, 0] - point[0]
        dlon = others[:, 1] - point[1]
        A = np.sin(dlat / 2.0) ** 2 + np.cos(point[0]) * np.cos(others[:, 0]) * np.sin(dlon / 2.0) ** 2
        return 6371.0 * 2.0 * np.arcsin(np.sqrt(A))  # km

    G = np.zeros(len(grid_xy), dtype=np.float32)
    for i, gx in enumerate(grid_rad):
        d = haversine(gx, prop_rad)
        w = masses * np.exp(-beta * d)
        G[i] = w.sum()
    return G

def predict_future_masses(df_listings, final_model, scaler, feature_cols):
    """
    Predice m_j(t+h) en espacio log_masa y luego lo devuelve en escala de masa.
    """
    Xr = df_listings[feature_cols].values.astype(np.float32)
    Xs = (Xr - scaler.mean_) / scaler.scale_
    Xt = torch.tensor(Xs, dtype=torch.float32).to(device)
    final_model.eval()
    with torch.no_grad():
        pred_log = final_model(Xt).cpu().numpy().flatten()
    # Volvemos a escala de masa
    m_future = np.expm1(pred_log)
    return m_future

def exceedance_probability(G_now, G_future, tau=0.10, B=50, noise_std=0.03):
    """
    Calcula P(ΔG > tau) con un ensemble simple.
    """
    delta = G_future - G_now
    probs = []

    for b in range(B):
        noise = np.random.normal(0.0, noise_std, size=len(delta))
        probs.append((delta + noise > tau).astype(np.float32))
    probs = np.stack(probs, axis=0).mean(axis=0)

    return probs, delta

def predict_gravity_growth(df_listings,
                           final_model,
                           scaler,
                           feature_cols,
                           grid_xy,
                           tau=0.10,
                           beta=0.12):
    """
    Integra todo:
    1) Usa masas actuales (columna masa_actual).
    2) Predice masas futuras con la red.
    3) Calcula G_now y G_future.
    4) Calcula prob. de excedencia y clasifica.
    Retorna un DataFrame con una fila por celda del grid.
    """

    prop_xy = df_listings[["lat", "lon"]].values.astype(np.float32)

    if "masa_actual" in df_listings.columns:
        m_now = df_listings["masa_actual"].values.astype(np.float32)
    else:
        raise ValueError("Se esperaba columna 'masa_actual' en df_listings.")

    m_future = predict_future_masses(df_listings, final_model, scaler, feature_cols)

    G_now = compute_gravity_field(
        grid_xy=grid_xy,
        prop_xy=prop_xy,
        masses=m_now,
        beta=beta
    )

    G_future = compute_gravity_field(
        grid_xy=grid_xy,
        prop_xy=prop_xy,
        masses=m_future,
        beta=beta
    )

    probs, delta = exceedance_probability(
        G_now=G_now,
        G_future=G_future,
        tau=tau,
        B=50,
        noise_std=0.03
    )

    clases = np.where(
        probs >= 0.7, "alta",
        np.where(probs >= 0.4, "media", "incipiente")
    )

    out = pd.DataFrame({
        "lat": grid_xy[:, 0],
        "lon": grid_xy[:, 1],
        "G_now": G_now,
        "G_future": G_future,
        "delta_G": delta,
        "prob_excede": probs,
        "clase": clases
    })

    return out

# ===========================================================
# 6. Función para generar KML con clases territoriales
# ===========================================================

def export_kml_zonas_clase(df_grid, kml_path, name_doc="Gravedad futura"):
    """
    Genera un KML con polígonos por celda del grid, agrupados por clase.
    df_grid debe tener columnas: lat, lon, clase, G_now, G_future, delta_G, prob_excede
    """

    # Extraemos latitudes y longitudes únicas y ordenadas
    lats = np.sort(df_grid["lat"].unique())
    lons = np.sort(df_grid["lon"].unique())
    n_lat, n_lon = len(lats), len(lons)

    # Construimos los bordes de las celdas a partir de los centros
    lat_edges = np.zeros(n_lat + 1)
    lon_edges = np.zeros(n_lon + 1)

    # bordes internos como promedio de centros consecutivos
    lat_edges[1:-1] = (lats[:-1] + lats[1:]) / 2.0
    lon_edges[1:-1] = (lons[:-1] + lons[1:]) / 2.0

    # bordes externos extrapolando el espaciamiento
    lat_step = lats[1] - lats[0] if n_lat > 1 else 0.001
    lon_step = lons[1] - lons[0] if n_lon > 1 else 0.001
    lat_edges[0] = lats[0] - lat_step / 2.0
    lat_edges[-1] = lats[-1] + lat_step / 2.0
    lon_edges[0] = lons[0] - lon_step / 2.0
    lon_edges[-1] = lons[-1] + lon_step / 2.0

    # Mapeo clase a estilos de color KML (aabbggrr)
    style_colors = {
        "alta": "7d0000ff",       # rojo
        "media": "7d00a5ff",      # naranja
        "incipiente": "7d00ff00"  # verde
    }

    # Cabecera de KML
    kml_parts = []
    kml_parts.append('<?xml version="1.0" encoding="UTF-8"?>')
    kml_parts.append('<kml xmlns="http://www.opengis.net/kml/2.2">')
    kml_parts.append("<Document>")
    kml_parts.append(f"<name>{name_doc}</name>")

    # Definimos estilos
    for cls, color in style_colors.items():
        kml_parts.append(f'<Style id="{cls}">')
        kml_parts.append("<PolyStyle>")
        kml_parts.append(f"<color>{color}</color>")
        kml_parts.append("<outline>0</outline>")
        kml_parts.append("</PolyStyle>")
        kml_parts.append("</Style>")

    # Carpeta por clase
    for cls in ["alta", "media", "incipiente"]:
        sub = df_grid[df_grid["clase"] == cls]
        if sub.empty:
            continue

        kml_parts.append(f"<Folder><name>{cls}</name>")

        # generamos una celda por fila
        for _, row in sub.iterrows():
            lat_c = row["lat"]
            lon_c = row["lon"]

            # localizamos índices de la grilla
            i_lat = np.searchsorted(lats, lat_c)
            i_lon = np.searchsorted(lons, lon_c)

            lat_min = lat_edges[i_lat]
            lat_max = lat_edges[i_lat + 1]
            lon_min = lon_edges[i_lon]
            lon_max = lon_edges[i_lon + 1]

            nombre = f"{cls} GΔ={row['delta_G']:.3f} P={row['prob_excede']:.2f}"

            coords = [
                f"{lon_min},{lat_min},0",
                f"{lon_max},{lat_min},0",
                f"{lon_max},{lat_max},0",
                f"{lon_min},{lat_max},0",
                f"{lon_min},{lat_min},0"
            ]

            kml_parts.append("<Placemark>")
            kml_parts.append(f"<name>{nombre}</name>")
            kml_parts.append(f"<styleUrl>#{cls}</styleUrl>")
            kml_parts.append("<Polygon>")
            kml_parts.append("<outerBoundaryIs>")
            kml_parts.append("<LinearRing>")
            kml_parts.append("<coordinates>")
            kml_parts.append(" ".join(coords))
            kml_parts.append("</coordinates>")
            kml_parts.append("</LinearRing>")
            kml_parts.append("</outerBoundaryIs>")
            kml_parts.append("</Polygon>")
            kml_parts.append("</Placemark>")

        kml_parts.append("</Folder>")

    kml_parts.append("</Document>")
    kml_parts.append("</kml>")

    kml_str = "\n".join(kml_parts)
    with open(kml_path, "w", encoding="utf-8") as f:
        f.write(kml_str)
    print(f"KML guardado en {kml_path}")

# ===========================================================
# 7. Ejemplo de uso completo
# ===========================================================

# Definimos un grid alrededor de la zona de estudio (Ajijic)
lat_min, lat_max = df_model["lat"].min() - 0.01, df_model["lat"].max() + 0.01
lon_min, lon_max = df_model["lon"].min() - 0.01, df_model["lon"].max() + 0.01

# Resolución del grid (ajusta según escala micro o meso)
lat_space = np.linspace(lat_min, lat_max, 60)
lon_space = np.linspace(lon_min, lon_max, 60)

grid = np.array([[la, lo] for la in lat_space for lo in lon_space])

resultado = predict_gravity_growth(
    df_listings=df_model,
    final_model=final_model,
    scaler=scaler,
    feature_cols=feature_cols,
    grid_xy=grid,
    tau=0.12,   # umbral de crecimiento de gravedad
    beta=0.15   # decaimiento espacial
)

os.makedirs("artefactos", exist_ok=True)
csv_path = "artefactos/gravedad_futura_grid.csv"
resultado.to_csv(csv_path, index=False)
print(f"Mapa de gravedad futura guardado en {csv_path}")

kml_path = "artefactos/gravedad_futura_clases.kml"
export_kml_zonas_clase(resultado, kml_path, name_doc="Crecimiento de gravedad territorial")


In [ ]:
# ============================================================================
# CONSTRUCCIÓN AUTOMÁTICA DE pairs.csv PARA RED DE ATRACCIÓN Aij
# ============================================================================

import numpy as np
import pandas as pd
from itertools import combinations
from math import radians, sin, cos, asin, sqrt

# ------------------------------
# 1) Cargar tu dataset real
# ------------------------------
# Usar df_model que ya está limpio y alineado con los embeddings
# Aseguramos que los IDs y las masas sean consistentes con df_model de Bloque 1
df_for_pairs_creation = df_model.copy()
df_for_pairs_creation["id"] = df_for_pairs_creation["id"].astype(str)
# Recuperar la masa original de la log_masa para el cálculo de Aij
df_for_pairs_creation["masa"] = np.expm1(df_for_pairs_creation["log_masa"])

# ------------------------------
# 2) Función de distancia haversine en km
# ------------------------------
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return 2 * R * asin(sqrt(a))

# ------------------------------
# 3) Parámetros del modelo gravitacional
# ------------------------------
beta = 0.12   # decaimiento espacial que ya usas en bloque 3
kmin = 1e-3   # distancia mínima para evitar división por 0

rows = []

# ------------------------------
# 4) Construir todas las parejas (combinations)
# ------------------------------
for i_idx, j_idx in combinations(df_for_pairs_creation.index, 2):
    row_i = df_for_pairs_creation.loc[i_idx]
    row_j = df_for_pairs_creation.loc[j_idx]

    dij = haversine(row_i["lat"], row_i["lon"], row_j["lat"], row_j["lon"])
    dij = max(dij, kmin)

    # fricción aproximada en minutos (puede ajustarse luego con velocidades reales)
    c_ij_minutos = dij / 0.20   # asumiendo 12 km ≈ 60 min => 1 km ≈ 5 min

    # Atracción con tu modelo base
    Aij = row_i["masa"] * row_j["masa"] * np.exp(-beta * dij)

    rows.append({
        "id_i": row_i["id"],
        "id_j": row_j["id"],
        "d_ij_km": dij,
        "c_ij_minutos": c_ij_minutos,
        "A_ij": Aij,
        "similaridad_semantica": 0.0  # opcional, 0 por ahora
    })

# duplicamos i,j y j,i para que sea dirigido si lo necesitas
df_pairs = pd.DataFrame(rows)
df_pairs_sym = pd.concat([
    df_pairs,
    df_pairs.rename(columns={"id_i": "id_j", "id_j": "id_i"})
], ignore_index=True)

df_pairs_sym.to_csv("pairs.csv", index=False)
print("pairs.csv generado con éxito.")

NameError: name 'df_model' is not defined